In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio
import shapely
import pyproj
import xarray as xr
import fiona
import pyogrio
import matplotlib as plt
import geodatasets
import folium
from shapely.geometry import Polygon
from shapely.geometry import box
import matplotlib.cm as cm
import matplotlib.colors as colors
import branca.colormap as bc

In [2]:
congo_basin_boundries = gpd.read_file("data/Congobasin_perimeter/SPCB_Report_Perimeter.gdb", layer="SPCB_Boundary")
congo_basin_boundries.to_crs(epsg='4326')
congo_fauna_biodiversity = gpd.read_file('congo_fauna_biodiversity.gpkg').to_crs(epsg='4326')

In [ ]:
#Create a grid over Congo Basin
xmin, ymin, xmax, ymax = congo_basin_boundries.total_bounds
cell_size = 0.25  # degrees (adjust later: 0.5, 0.25, etc.)

grid_cells = [
    box(x, y, x + cell_size, y + cell_size)
    for x in np.arange(xmin, xmax, cell_size)
    for y in np.arange(ymin, ymax, cell_size)
]

grid = gpd.GeoDataFrame(geometry=grid_cells, crs="EPSG:4326")

In [ ]:
grid = gpd.overlay(grid, congo_basin_boundries, how="intersection")

In [ ]:
join = gpd.sjoin(grid, congo_fauna_biodiversity, predicate="intersects")

In [ ]:
# Add a presence indicator
join["presence"] = 1

# Convert from long format to wide format
matrix = join.pivot_table(
    index="index_left",      # one row per grid cell
    columns="binomial",      # one column per species
    values="presence",       # use the presence values
    fill_value=0             # 0 = species absent
)

# Ensure values are integers (0 or 1)
matrix = matrix.astype(int)

   FID  ID                                             MOSAIC MAJOR_VEG_  \
0    1   1  Mediterranean sclerophyllous forest (s) Quercu...     FOREST   
1    2   2  Mediterranean sclerophyllous forest (s) Quercu...     FOREST   
2    3   3  Mediterranean sclerophyllous forest (s) Quercu...     FOREST   
3    4   4  Mediterranean sclerophyllous forest (s) Quercu...     FOREST   
4    5   5  Mediterranean sclerophyllous forest (s) Quercu...     FOREST   

    Shape__Area  Shape__Length  \
0  3.249445e+08   86302.691775   
1  3.222587e+08   82833.174668   
2  9.530747e+07   46212.360600   
3  3.303088e+08   71153.959514   
4  4.017219e+09  361914.888041   

                                            geometry  
0  POLYGON ((8.99167 37.05417, 9.04999 37.12219, ...  
1  POLYGON ((11.10833 36.80417, 11.10493 36.8091,...  
2  POLYGON ((2.75 36.6375, 2.76157 36.65093, 2.86...  
3  POLYGON ((3.26667 36.77083, 3.27288 36.81631, ...  
4  POLYGON ((7.75833 36.9375, 7.82151 36.94266, 7...  
(1013, 7

Index(['FID', 'ID', 'MOSAIC', 'MAJOR_VEG_', 'Shape__Area', 'Shape__Length',
       'geometry'],
      dtype='object')
